In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

def calculate_changes(orig_row, cf_row, features, threshold=0.01):
    """Calculate number of changes between original and counterfactual rows."""
    changes = []
    for feat in features:
        if feat in ['Sex', 'Race']:  # Categorical features
            changes.append(orig_row[feat] != cf_row[feat])
        else:  # Numerical features
            normalized_diff = abs(orig_row[feat] - cf_row[feat]) / df[feat].std()
            changes.append(normalized_diff > threshold)
    return sum(changes)

def export_counterfactual_analysis(original_rows, counterfactual_rows, features, distances, changes, output_file='counterfactual_analysis_results.csv'):
    """Export counterfactual analysis results to a CSV file."""
    # Calculate overall metrics
    avg_distance = np.mean(distances)
    std_distance = np.std(distances)
    avg_changes = np.mean(changes)
    std_changes = np.std(changes)
    
    # Calculate feature-specific change rates
    feature_change_rates = {}
    for feat in features:
        if feat in ['Sex', 'Race']:  # Categorical features
            changes = (original_rows[feat].values != counterfactual_rows[feat].values).mean() * 100
        else:  # Numerical features
            normalized_diff = abs(original_rows[feat].values - counterfactual_rows[feat].values) / pd.concat([original_rows[feat], counterfactual_rows[feat]]).std()
            changes = (normalized_diff > 0.01).mean() * 100
        feature_change_rates[feat] = changes
    
    # Create DataFrames for each section
    overall_metrics = pd.DataFrame({
        'Metric': [
            'Normalized Average Distance (Mean)',
            'Normalized Average Distance (Std)',
            'Average Feature Changes (Mean)',
            'Average Feature Changes (Std)',
            'Percentage of Features Changed'
        ],
        'Value': [
            f'{avg_distance:.3f}',
            f'{std_distance:.3f}',
            f'{avg_changes:.3f}',
            f'{std_changes:.3f}',
            f'{(avg_changes/len(features))*100:.1f}%'
        ]
    })
    
    feature_metrics = pd.DataFrame({
        'Feature': list(feature_change_rates.keys()),
        'Change Rate (%)': [f'{rate:.1f}%' for rate in feature_change_rates.values()]
    }).sort_values('Change Rate (%)', ascending=False)
    
    # Add section headers and combine
    overall_metrics.insert(0, 'Section', 'Overall Metrics')
    feature_metrics.insert(0, 'Section', 'Feature-Specific Analysis')
    
    # Combine all results
    combined_results = pd.concat([overall_metrics, feature_metrics], ignore_index=True)
    
    # Export to CSV
    combined_results.to_csv(output_file, index=False)
    
    return combined_results

# Load the data
df = pd.read_csv('counterfactual_pairs_all_with_outcomes.csv')

# Convert index column to string if it exists
if 'index' in df.columns:
    df['index'] = df['index'].astype(str)
    # Separate original and counterfactual examples
    original_rows = df[df['index'] == 'Original']
    counterfactual_rows = df[df['index'] == 'Counterfactual']
else:
    # Assume alternating rows if no index column
    original_rows = df.iloc[::2]  # Every even row
    counterfactual_rows = df.iloc[1::2]  # Every odd row

# Features to consider (excluding 'Outcome' and index-related columns)
features = ['Sex', 'Race', 'Age', 'Income', 'WaistCirc', 'BMI', 'Albuminuria', 
           'UrAlbCr', 'UricAcid', 'BloodGlucose', 'HDL', 'Triglycerides']

# Initialize and fit StandardScaler
scaler = StandardScaler()
all_features = df[features].values
scaler.fit(all_features)

# Transform the data
original_normalized = scaler.transform(original_rows[features])
counterfactual_normalized = scaler.transform(counterfactual_rows[features])

# Calculate normalized distances
distances = np.linalg.norm(original_normalized - counterfactual_normalized, axis=1)

# Calculate number of changes for each pair
changes = []
for i in range(len(original_rows)):
    orig = original_rows.iloc[i]
    cf = counterfactual_rows.iloc[i]
    num_changes = calculate_changes(orig, cf, features)
    changes.append(num_changes)

# Export results to CSV and display summary
results_df = export_counterfactual_analysis(
    original_rows=original_rows,
    counterfactual_rows=counterfactual_rows,
    features=features,
    distances=distances,
    changes=changes
)

# Print results to console
print("\nCounterfactual Analysis Results:")
print("-" * 50)
print("\nOverall Metrics:")
print(results_df[results_df['Section'] == 'Overall Metrics'][['Metric', 'Value']].to_string(index=False))
print("\nFeature-Specific Analysis:")
print(results_df[results_df['Section'] == 'Feature-Specific Analysis'][['Feature', 'Change Rate (%)']].to_string(index=False))
print("\nResults have been saved to 'counterfactual_analysis_results.csv'")